# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook provides a guided workflow for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source

The dataset is defined by a [Croissant schema](https://mlcommons.org/croissant/) at the following URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`


In [ ]:
# Ensure mlcroissant is installed in your environment
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the Croissant dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset package
dataset = mlc.Dataset(croissant_url)

# Access top-level metadata attributes
meta = dataset.metadata
print(f"{meta.name}: {meta.description}")


## 2. Data Overview

List all available `RecordSet` entities in the dataset, along with their `@id`s, and for each, list their fields and columns using their `@id`s for reference.

In [ ]:
# List all RecordSets in the dataset and summarize their Fields by @id
record_sets = dataset.record_sets
print(f"Found {len(record_sets)} record sets:")

for rs in record_sets:
    print(f"\nRecord set: {rs['@id']}  -- name: {rs.get('name', '---')}")
    fields = rs.get('field', [])
    # Ensure fields is a list
    if isinstance(fields, dict):
        fields = [fields]
    print("  Fields (by @id):")
    for f in fields:
        print(f"    - {f['@id']} (name: {f.get('name', '---')})")
        columns = f.get('column', [])
        if isinstance(columns, dict):
            columns = [columns]
        if columns:
            print(f"      Columns (by @id):")
            for col in columns:
                print(f"        - {col['@id']} (name: {col.get('name', '---')})")


## 3. Data Extraction

Load data from the main record set into a DataFrame for analysis. We'll use the record set and field `@id`s identified above.

The main record set of interest, containing clinicopathological data, is likely the only (or first) tabular record set.

In [ ]:
# Find all tabular record set IDs
tabular_record_set_ids = []
for rs in dataset.record_sets:
    if rs.get('@type') == 'cr:RecordSet' or 'RecordSet' in rs.get('@type', ''):
        tabular_record_set_ids.append(rs['@id'])

# If there are no 'cr:RecordSet' types found, just take all record sets
if not tabular_record_set_ids:
    tabular_record_set_ids = [rs['@id'] for rs in dataset.record_sets]

# Display the RecordSet IDs available
print("Tabular RecordSet @id(s) available:")
for rid in tabular_record_set_ids:
    print(f" - {rid}")

# Load all data from the tabular record sets
dataframes = {}
for record_set_id in tabular_record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)

# Display the columns in the primary DataFrame (first record set)
main_rs_id = tabular_record_set_ids[0]
print(f"\nColumns in main record set {main_rs_id}:")
print(dataframes[main_rs_id].columns.tolist())
dataframes[main_rs_id].head()

## 4. Exploratory Data Analysis (EDA)

We'll demonstrate common EDA steps such as:
- Filtering (for example, isolating records where a numeric field exceeds a threshold)
- Normalization of a numeric field
- Grouping by a categorical column

**Note:**
All columns and fields are referenced by their `@id`, per the FAIR²/Croissant conventions.

First, let's print all the columns and select an appropriate numeric and group field by their `@id`.

In [ ]:
# Review columns in the main DataFrame
df = dataframes[main_rs_id]
print("Column (@id)s in main record set:")
for idx, col in enumerate(df.columns):
    print(f" {idx}: {col}")

# MANUAL SELECTION (edit as appropriate to your data):
# Let's suppose the field @id for 'Interval_between_first_and_second_cancer_years' is:
numeric_field_id = None
group_field_id = None
for col in df.columns:
    # Try to pick a numeric interval or age column
    if 'interval' in col.lower() or 'age' in col.lower():
        numeric_field_id = col
    if not group_field_id and (
        'sex' in col.lower() or 'msi' in col.lower() or 'location' in col.lower()):
        group_field_id = col

print(f"\nUsing numeric field: {numeric_field_id}")
print(f"Using group field: {group_field_id}")

if numeric_field_id is not None:
    # Clean non-numeric data if needed
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

    # Set threshold (e.g., mean or a fixed value)
    threshold = df[numeric_field_id].mean()

    # Filtering step
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"\nFiltered records with {numeric_field_id} above mean (threshold = {threshold:.2f}):")
    print(filtered_df.head())

    # Normalization
    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
        filtered_df[numeric_field_id].std()
    )
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Grouping
    if group_field_id is not None and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nGrouped (mean of {numeric_field_id}) by {group_field_id}:")
        print(grouped_df.head())
else:
    print("No numeric field was found for EDA demonstration.")

## 5. Visualization

We will visualize the distribution of the chosen numeric field and, if possible, the mean value across categories in the selected group field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot distribution of numeric field
if numeric_field_id is not None:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=15)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # If grouping is possible, show mean by group
    if group_field_id is not None and group_field_id in df.columns:
        plt.figure(figsize=(8,4))
        sns.barplot(y=group_field_id, x=numeric_field_id, data=df, ci=None, estimator=pd.Series.mean)
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.show()

## 6. Conclusion

In this notebook, we demonstrated how to:
- Load and parse a Croissant schema dataset using `mlcroissant`.
- List record sets, fields, and columns by their `@id`, as required by the FAIR² and Croissant standards.
- Extract and analyze the content of the main study record set representing clinicopathological and molecular features of second primary colorectal cancer in cancer survivors.
- Perform exploratory data analysis: filtering, normalization, grouping, and basic visualization of key numeric fields.

This workflow can be customized for other Croissant datasets for further FAIR clinical and biomedical data science applications.